# LAB 02 — K-Nearest Neighbors: từ khoảng cách đến quyết định

**DNU Learning Analytics Lab**

- **Họ tên:** ...
- **MSSV:** ...
- **Lớp:** ...

> **Câu hỏi trung tâm:** KNN đưa ra dự đoán dựa trên “hàng xóm” như thế nào, và điều gì làm thay đổi những hàng xóm đó?

Lab bắt đầu bằng bài toán A/B tổng quát rồi mới quay lại `StudentPerformanceFactors`, để nhấn mạnh KNN là thuật toán tổng quát.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, ConfusionMatrixDisplay
from knn_manual import euclidean_distance, manhattan_distance, get_k_neighbors, knn_predict_one
DATA_PATH = Path('data/StudentPerformanceFactors.csv')
print('Environment ready! Dataset exists:', DATA_PATH.exists())

# Mission 1 — Distance first: “gần” nghĩa là gì?

Với `A=(3,3)` và `B=(5,5)`, hãy tính bằng tay Euclidean và Manhattan distance rồi hoàn thiện hai hàm tương ứng trong `knn_manual.py`.

**[TRẢ LỜI]** Euclidean = ...; Manhattan = ...

In [ ]:
A, B = [3,3], [5,5]
for name, fn in [('Euclidean', euclidean_distance), ('Manhattan', manhattan_distance)]:
    try:
        print(name, '=', fn(A, B))
    except NotImplementedError as e:
        print(name, '-> TODO:', e)

## Tình huống đổi metric thì đổi prediction

Với `sample=(5,5)`, `K=3` và năm điểm A–E dưới đây, chỉ đổi Euclidean sang Manhattan có thể làm nhãn dự đoán đổi mà không hòa phiếu. Hãy dự đoán trước khi chạy code.

| Điểm | x1 | x2 | Nhãn |
|---|---:|---:|---|
| A | 6 | 6 | Class A |
| B | 8 | 8 | Class A |
| C | 2 | 8 | Class A |
| D | 5 | 10 | Class B |
| E | 10 | 5 | Class B |

**[TRẢ LỜI]** ...

In [ ]:
X_generic = np.array([[6,6],[8,8],[2,8],[5,10],[10,5]], dtype=float)
y_generic = np.array(['Class A','Class A','Class A','Class B','Class B'])
sample_generic = np.array([5,5], dtype=float)
for metric in ['euclidean','manhattan']:
    try:
        print(metric, '->', knn_predict_one(X_generic, y_generic, sample_generic, k=3, metric=metric))
    except NotImplementedError as e:
        print(metric, '-> TODO:', e)

# Mission 2 — Build KNN from scratch

Hoàn thiện `get_k_neighbors()` và `knn_predict_one()` trong `knn_manual.py`. Quy trình: **distance → sort → K neighbors → majority vote → prediction**. Không dùng `sklearn.neighbors` trong phần này.

Sau đó chạy `python -m pytest -q tests/test_knn_manual.py`.

**[TRẢ LỜI]** KNN thật sự cần những thông tin gì để dự đoán?

In [ ]:
try:
    print(get_k_neighbors(X_generic, y_generic, sample_generic, k=3, metric='euclidean'))
except NotImplementedError as e:
    print('TODO in knn_manual.py ->', e)
plt.figure(figsize=(6,5))
for label in ['Class A','Class B']:
    m = y_generic == label
    plt.scatter(X_generic[m,0], X_generic[m,1], s=90, label=label)
plt.scatter(*sample_generic, marker='*', s=250, label='New sample')
plt.xlabel('Feature 1'); plt.ylabel('Feature 2'); plt.legend(); plt.show()

# Mission 3 — Back to DNU case study

Tạo nhãn giảng dạy `Needs_Support = 1` nếu `Exam_Score < 65`, ngược lại là 0. Đây chỉ là nhãn giả lập phục vụ học tập, không phải quy định chính thức của DNU.

**Quan trọng:** không dùng `Exam_Score` làm feature vì sẽ gây target leakage.

**[TRẢ LỜI]** Vì sao đây là leakage? Các feature hiện tại có cùng thang đo không?

In [ ]:
if not DATA_PATH.exists():
    raise FileNotFoundError('Run: python scripts/download_data.py')
df = pd.read_csv(DATA_PATH).copy()
df['Needs_Support'] = (df['Exam_Score'] < 65).astype(int)
feature_cols = ['Hours_Studied','Attendance','Previous_Scores','Sleep_Hours']
assert 'Exam_Score' not in feature_cols
model_df = df[feature_cols + ['Needs_Support']].dropna().copy()
X = model_df[feature_cols]
y = model_df['Needs_Support']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
print('Train:', X_train.shape, 'Test:', X_test.shape)
display(y.value_counts())

# Mission 4 — Baseline KNN: chưa scaling

Huấn luyện `KNeighborsClassifier` với `K=5`, Euclidean distance. Đọc accuracy, precision, recall và confusion matrix theo bối cảnh `Needs_Support`.

**[TRẢ LỜI]** FN là trường hợp nào? FP là trường hợp nào? Bạn lo lỗi nào hơn và vì sao?

In [ ]:
def evaluate(name, model, X_eval, y_eval):
    pred = model.predict(X_eval)
    result = {'accuracy': accuracy_score(y_eval,pred), 'precision': precision_score(y_eval,pred,zero_division=0), 'recall': recall_score(y_eval,pred,zero_division=0)}
    print(name, result)
    ConfusionMatrixDisplay.from_predictions(y_eval, pred, display_labels=['No Support','Needs Support'])
    plt.title(name); plt.show()
    return result, pred
baseline_knn = KNeighborsClassifier(n_neighbors=5, metric='euclidean')
baseline_knn.fit(X_train, y_train)
baseline_result, baseline_pred = evaluate('KNN baseline — unscaled', baseline_knn, X_test, y_test)

# Mission 5 — Scaling matters

Fit `StandardScaler` **chỉ trên train set**, rồi so sánh mô hình trước/sau scaling. Quan trọng hơn: quan sát hàng xóm của cùng một sample có thay đổi không.

**[TRẢ LỜI]** Scaling làm thay đổi khái niệm ‘gần’ như thế nào?

In [ ]:
display(pd.DataFrame({'min':X_train.min(),'max':X_train.max(),'range':X_train.max()-X_train.min()}))
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
scaled_knn = KNeighborsClassifier(n_neighbors=5, metric='euclidean')
scaled_knn.fit(X_train_scaled, y_train)
scaled_result, scaled_pred = evaluate('KNN — StandardScaler', scaled_knn, X_test_scaled, y_test)
sample_raw = X_test.iloc[[0]]
sample_scaled = X_test_scaled[[0]]
raw_dist, raw_idx = baseline_knn.kneighbors(sample_raw, n_neighbors=5)
scaled_dist, scaled_idx = scaled_knn.kneighbors(sample_scaled, n_neighbors=5)
print('Raw neighbor train positions:', raw_idx[0].tolist())
print('Scaled neighbor train positions:', scaled_idx[0].tolist())
print('Prediction raw/scaled:', baseline_knn.predict(sample_raw)[0], scaled_knn.predict(sample_scaled)[0])

# Mission 6 — K and distance metric matter

Không dùng `GridSearchCV`. Thử thủ công `K ∈ {1,3,5,7,9}` và metric `euclidean`, `manhattan` trên dữ liệu đã scaling.

**[TRẢ LỜI]** K tốt nhất theo accuracy có giống K tốt nhất theo recall không? Nếu mục tiêu là không bỏ sót sinh viên cần hỗ trợ, nên ưu tiên metric đánh giá nào?

In [ ]:
rows=[]
for metric in ['euclidean','manhattan']:
    for k in [1,3,5,7,9]:
        model=KNeighborsClassifier(n_neighbors=k, metric=metric)
        model.fit(X_train_scaled,y_train)
        pred=model.predict(X_test_scaled)
        rows.append({'metric':metric,'k':k,'accuracy':accuracy_score(y_test,pred),'precision':precision_score(y_test,pred,zero_division=0),'recall':recall_score(y_test,pred,zero_division=0)})
results_df=pd.DataFrame(rows)
display(results_df)
plt.figure(figsize=(8,5))
for metric in ['euclidean','manhattan']:
    t=results_df[results_df.metric==metric]
    plt.plot(t.k,t.accuracy,marker='o',label=metric)
plt.xlabel('K'); plt.ylabel('Accuracy'); plt.legend(); plt.show()

# Transfer Challenge — KNN không chỉ dành cho Student Performance

Dùng Iris dataset có sẵn trong sklearn. Tự xây dựng pipeline: load → split → scaling → KNN → evaluation.

> **Những bước nào của KNN không thay đổi khi đối tượng chuyển từ sinh viên sang hoa?**

**[TRẢ LỜI]** ...

In [ ]:
from sklearn.datasets import load_iris
iris = load_iris(as_frame=True)
X_iris, y_iris = iris.data, iris.target
# TODO: train_test_split -> StandardScaler -> KNeighborsClassifier -> accuracy
print('Iris shape:', X_iris.shape, 'Classes:', iris.target_names)

# Final — Model Reflection Card

## 1. KNN dự đoán như thế nào?
**[VIẾT]**

## 2. Ba yếu tố làm prediction thay đổi
- **K:** [VIẾT]
- **Distance metric:** [VIẾT]
- **Feature scaling:** [VIẾT]

## 3. Trong case study `Needs_Support`, lỗi nào đáng lo hơn?
**[VIẾT]**

## 4. Vì sao `Exam_Score` không được dùng làm feature?
**[VIẾT]**

## 5. Một câu chốt
> KNN không phải thuật toán dành riêng cho sinh viên vì...

**[VIẾT]**

## Trước khi push

```bash
python tests/check_lab02.py
python -m pytest -q tests/test_knn_manual.py
git add .
git commit -m "Complete Lab 02 KNN"
git push
```

Kiểm tra tab **Actions** và đảm bảo workflow `Lab 02 Check` chạy thành công sau khi hoàn thiện bài.